# MOD-02 · NB-01 — Data Loading & Automated Profiling
### Health Analytics with Python · Module 02: EDA in Healthcare
---
**Learning objectives**
- Load health data from CSV, Excel, and Parquet with production-grade options
- Generate an automated profiling report with `ydata-profiling`
- Interpret profiling output sections: overview, variables, correlations, missing values
- Customise and export the report for sharing with clinical collaborators

**Estimated time:** 1 hour | **Level:** Beginner | **Libraries:** `pandas`, `ydata-profiling`, `openpyxl`, `pyarrow`


## 1. Setup & Dependency Check

In [ ]:
# Install ydata-profiling if needed (comment out after first run)
# !pip install ydata-profiling openpyxl pyarrow

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    from ydata_profiling import ProfileReport
    from ydata_profiling.config import Settings
    Settings().plot.font_path = '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
    print("ydata-profiling imported successfully.")
except ImportError:
    print("Run: pip install ydata-profiling")

print(f"pandas version: {pd.__version__}")


## 2. Synthetic Hospital Discharge Dataset

In [ ]:
np.random.seed(42)
N = 800

ages      = np.random.normal(62, 16, N).clip(18, 95).astype(int)
sexes     = np.random.choice(['M','F'], N, p=[0.48,0.52])
payers    = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],
                              N, p=[0.40,0.22,0.28,0.07,0.03])
drg_codes = np.random.choice([470,291,392,690,871,193,247,603], N)
drg_labels = {470:'Major joint replacement',291:'Heart failure & shock',
              392:'Esophagitis/misc GI',690:'Kidney/UTI',871:'Septicemia',
              193:'Simple pneumonia',247:'Perc cardiovasc w stent',603:'Cellulitis'}
los_days  = np.random.gamma(2.5, 1.8, N).clip(1, 30).astype(int)
charges   = (los_days * np.random.uniform(1800, 4200, N)).round(2)
charges[np.random.rand(N) < 0.08] = np.nan
primary_dx = np.random.choice(
    ['E11.9','I10','N18.3','I50.9','J18.9','A41.9','M16.11','N39.0'], N)
admit_type = np.random.choice(
    ['Emergency','Elective','Urgent','Newborn'], N, p=[0.52,0.30,0.16,0.02])
np.random.seed(99)
df = pd.DataFrame({
    'patient_id'   : [f'PT-{str(i).zfill(5)}' for i in range(1, N+1)],
    'age'          : ages, 'sex': sexes, 'payer': payers,
    'drg_code'     : drg_codes,
    'drg_label'    : [drg_labels[d] for d in drg_codes],
    'primary_dx'   : primary_dx, 'admit_type': admit_type,
    'los_days'     : los_days,   'total_charge': charges,
    'diabetes'     : np.random.binomial(1,0.32,N),
    'hypertension' : np.random.binomial(1,0.45,N),
    'ckd'          : np.random.binomial(1,0.15,N),
    'heart_failure': np.random.binomial(1,0.18,N),
    'readmitted_30': np.random.binomial(1,0.14,N),
})
bins   = [17,44,64,74,95]
df['age_group'] = pd.cut(df['age'], bins=bins, labels=['18-44','45-64','65-74','75+'])

start = pd.Timestamp('2019-01-01')
np.random.seed(5)
df['admit_date'] = [start + pd.Timedelta(days=int(d))
                    for d in np.random.randint(0,(pd.Timestamp('2023-12-31')-start).days, N)]
df['discharge_date'] = df['admit_date'] + pd.to_timedelta(df['los_days'], unit='D')
df = df.sort_values('admit_date').reset_index(drop=True)
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head(4)


## 3. Saving & Re-loading in Multiple Formats

In [ ]:
Path('/tmp/mod02').mkdir(exist_ok=True)

# CSV — most universal
df.to_csv('/tmp/mod02/discharges.csv', index=False)

# Excel — useful for sharing with clinical stakeholders
df.to_excel('/tmp/mod02/discharges.xlsx', index=False, sheet_name='Discharges')

# Parquet — columnar, fast, preserves dtypes; preferred for large datasets
df.to_parquet('/tmp/mod02/discharges.parquet', index=False)

print("Files written. Sizes:")
for p in Path('/tmp/mod02').iterdir():
    print(f"  {p.name:30s}  {p.stat().st_size/1024:.1f} KB")


In [ ]:
# Re-load with production-grade options
df_csv = pd.read_csv(
    '/tmp/mod02/discharges.csv',
    dtype={'patient_id': str, 'drg_code': int},
    parse_dates=['admit_date','discharge_date'],
    low_memory=False   # avoids mixed-type warnings on large files
)

df_parquet = pd.read_parquet('/tmp/mod02/discharges.parquet')

print("CSV shape    :", df_csv.shape)
print("Parquet shape:", df_parquet.shape)
print("\nParquet dtypes (preserved automatically):")
print(df_parquet.dtypes)


## 4. Manual First-Pass Inspection

In [ ]:
# Always do this before profiling — catches obvious issues immediately
print("=== Shape ===")
print(df.shape)

print("\n=== dtypes ===")
print(df.dtypes)

print("\n=== Missing values ===")
miss = df.isnull().sum()
print(miss[miss > 0].to_frame('n_missing').assign(pct=lambda x: (x.n_missing/len(df)*100).round(1)))

print("\n=== Numeric summary ===")
display(df.describe(percentiles=[.10,.25,.50,.75,.90]).round(1))

print("\n=== Categorical cardinality ===")
for col in ['sex','payer','admit_type','primary_dx','drg_label']:
    print(f"  {col:15s}: {df[col].nunique():3d} unique")


## 5. Automated Profiling with ydata-profiling

In [ ]:
# Full profile — generates a comprehensive interactive HTML report
profile = ProfileReport(
    df,
    title       = "Hospital Discharge Dataset — Module 02 EDA",
    explorative = True,          # enables extra correlation checks
    minimal     = False,         # full report (set True for large datasets)
    correlations={               # only compute Pearson & Spearman
        'pearson' : {'calculate': True},
        'spearman': {'calculate': True},
        'kendall' : {'calculate': False},
        'phi_k'   : {'calculate': False},
        'cramers' : {'calculate': False},
    },
    missing_diagrams={
        'bar'     : True,
        'matrix'  : True,
        'heatmap' : True,
    },
    progress_bar=True,
)
print("Profile computed.")


In [ ]:
# Save as self-contained HTML (share with collaborators — no Python needed)
profile.to_file('/tmp/mod02/discharge_profile.html')
print("Report saved to /tmp/mod02/discharge_profile.html")

# Render inline in Jupyter
profile.to_notebook_iframe()


## 6. Interpreting the Profile Report

The report is divided into five sections. Here is what to look for in each.

| Section | What to check |
|---------|--------------|
| **Overview** | Number of variables, rows, missing cells, duplicate rows |
| **Variables** | Distribution shape, % missing, distinct count, warnings (high cardinality, skewed, zeros) |
| **Interactions** | Bivariate scatter plots — look for clusters or non-linear relationships |
| **Correlations** | Pearson (linear), Spearman (monotonic) — flag high correlations (>0.8) |
| **Missing values** | Bar chart + matrix — check if missingness clusters on specific patients or time periods |
| **Sample** | First/last rows — sanity check on values and date formatting |


In [ ]:
# --- Extract key stats programmatically from the profile object ---
desc = profile.get_description()
table = desc.table

print("=== Profile Overview ===")
print(f"  Dataset: {table['n']} rows, {table['n_var']} variables")
print(f"  Missing cells: {table['n_cells_missing']} ({table['p_cells_missing']*100:.1f}%)")
print(f"  Duplicate rows: {df.duplicated().sum()}")

print("\n=== Variable-level warnings ===")
for warn in desc.alerts:
    print(f"  [{warn.alert_type.name:20s}] {warn.column_name}")


## 7. Minimal Profile for Large Datasets

In [ ]:
# When dataset > 100k rows, use minimal=True for speed
profile_mini = ProfileReport(
    df,
    title   = "Discharge Profile (minimal)",
    minimal = True,
)
profile_mini.to_file('/tmp/mod02/discharge_profile_minimal.html')
print("Minimal profile saved.")

# Compare report sizes
for name in ['discharge_profile.html','discharge_profile_minimal.html']:
    size_kb = Path(f'/tmp/mod02/{name}').stat().st_size / 1024
    print(f"  {name:45s}: {size_kb:.0f} KB")


## 8. Knowledge Check

1. Why is Parquet preferred over CSV for large health datasets?
2. In the profile report, what does a 'HIGH CARDINALITY' warning mean for `patient_id`?
3. `total_charge` has 8% missing values. What should you investigate before imputing?
4. The profile shows Pearson correlation of 0.82 between `los_days` and `total_charge`. Is this expected? Why?
5. Re-run the profile on only the Medicare subgroup. Do the warnings change?


In [ ]:
# --- Exercise 5 ---
df_medicare = df[df['payer'] == 'Medicare'].copy()
profile_mc = ProfileReport(df_medicare, title="Medicare Subgroup Profile", minimal=True)
profile_mc.to_notebook_iframe()


---
## Summary
- Loaded health data from CSV, Excel, and Parquet with robust dtype handling
- Ran a manual first-pass inspection (shape, dtypes, missingness, cardinality)
- Generated a full `ydata-profiling` report and interpreted all five sections
- Extracted profile metadata programmatically for automated QC pipelines
- Used `minimal=True` for large-dataset performance

**Next → NB-02: Descriptive Statistics for Clinical Variables**
